---

# DSC Gelar Rasa 2026: Konversi Bobot DINOv3 ViT-7B ke fp16

*Notebook utilitas sekali jalan untuk menyiapkan dataset bobot fp16 yang dipakai notebook utama.*

---

## Nama Tim

- Nama anggota 1
- Nama anggota 2
- Nama anggota 3

---

## Daftar Isi

1. [**Pendahuluan**](#1)
2. [**Inisialisasi**](#2)
3. [**Konversi**](#3)
4. [**Verifikasi**](#4)


---

# Pendahuluan <a name="1"></a>

---

Bobot resmi `timm/vit_7b_patch16_dinov3.lvd1689m` disimpan dalam fp32 (26,9 GB). Notebook utama memakai bobot fp16 yang dibagi ke dua T4, jadi konversi sekali di sini menghemat unduhan dan ruang disk di setiap run. Notebook ini hanya mengubah dtype tensor dan tidak menjalankan model.

Cara pakai:

1. Jalankan dengan Internet ON (CPU sudah cukup).
2. Setelah selesai, buat dataset Kaggle **privat** dari folder `/kaggle/working/dinov3_7b_fp16`, sertakan teks DINOv3 License.
3. Lampirkan dataset itu ke notebook utama. Notebook utama mencari pola `dinov3_7b_fp16*.safetensors`.

---

# Inisialisasi <a name="2"></a>

---

## Environment Setup

Unduhan fp32 butuh sekitar 27 GB di `/tmp`, dan keluaran fp16 sekitar 13,4 GB di `/kaggle/working` (batas 20 GB). Sel berikut menampilkan ruang disk yang tersedia.

In [ ]:
!df -h /tmp /kaggle/working

Sel berikut memastikan `safetensors` dan `huggingface_hub` terpasang.

In [ ]:
%pip install -q safetensors huggingface_hub

## Import Libraries

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from safetensors.torch import save_file

## Settings

Revisi bobot dikunci supaya hasil konversi selalu sama dengan revisi yang dipakai notebook utama.

In [ ]:
class Settings:
    REPO = 'timm/vit_7b_patch16_dinov3.lvd1689m'
    REV = 'b40f6c827cb02b13c7a23a76cf598489bb0f94e5'
    HF_CACHE = Path('/tmp/hf')
    OUT_DIR = Path('/kaggle/working/dinov3_7b_fp16')
    SHARD_BYTES = 4 * 2**30
    FP16_MAX = 65504.0
    DISK_MIN_GIB = 30


CFG = Settings()
CFG.HF_CACHE.mkdir(parents=True, exist_ok=True)
CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
bebas = shutil.disk_usage(CFG.HF_CACHE).free / 2**30
assert bebas >= CFG.DISK_MIN_GIB, f'/tmp hanya {bebas:.0f} GiB, butuh minimal {CFG.DISK_MIN_GIB} GiB untuk bobot fp32'
print(f'/tmp bebas {bebas:.0f} GiB')

---

# Konversi <a name="3"></a>

---

Tensor dibaca satu per satu dengan `safe_open`, jadi RAM tidak pernah memuat seluruh bobot. Tensor float dikonversi ke fp16 dan dicek finite, lalu ditulis per shard 4 GiB. Buffer RoPE tidak ada di checkpoint (non-persisten), jadi tidak ikut terkonversi.

In [ ]:
src = hf_hub_download(CFG.REPO, 'model.safetensors', revision=CFG.REV, cache_dir=CFG.HF_CACHE)
manifest = {'repo': CFG.REPO, 'revision': CFG.REV, 'shard': {}}
shard, ukuran, n, maks = {}, 0, 0, 0.0


def simpan(shard, n):
    f = CFG.OUT_DIR / f'dinov3_7b_fp16-{n:05d}.safetensors'
    save_file(shard, str(f), metadata={'format': 'pt', 'sumber': f'{CFG.REPO}@{CFG.REV}'})
    manifest['shard'][f.name] = hashlib.sha256(f.read_bytes()).hexdigest()
    print(f.name, len(shard), 'tensor', flush=True)


with safe_open(src, 'pt', device='cpu') as sf:
    for k in sf.keys():
        t = sf.get_tensor(k)
        if t.is_floating_point():
            maks = max(maks, t.abs().max().item())
            t = t.to(torch.float16)
            assert torch.isfinite(t).all(), f'{k} overflow saat konversi ke fp16'
        shard[k] = t.contiguous()
        ukuran += t.numel() * t.element_size()
        if ukuran >= CFG.SHARD_BYTES:
            n += 1
            simpan(shard, n)
            shard, ukuran = {}, 0
if shard:
    n += 1
    simpan(shard, n)
manifest['maks_abs_bobot_fp32'] = maks
(CFG.OUT_DIR / 'manifest_dinov3_7b_fp16.json').write_text(json.dumps(manifest, indent=1))
print('selesai, |bobot| maks', maks)

---

# Verifikasi <a name="4"></a>

---

Hasil dicek terhadap sumber: jumlah tensor dan nama kunci harus sama persis, nilai bobot terbesar harus di bawah batas fp16, dan total ukuran sekitar 13,4 GB. Berkas fp32 di `/tmp` dihapus setelah lolos.

In [ ]:
kunci_fp16 = set()
for f in sorted(CFG.OUT_DIR.glob('dinov3_7b_fp16-*.safetensors')):
    with safe_open(str(f), 'pt', device='cpu') as sf:
        kunci_fp16 |= set(sf.keys())
with safe_open(src, 'pt', device='cpu') as sf:
    kunci_src = set(sf.keys())
assert kunci_fp16 == kunci_src, (len(kunci_fp16), len(kunci_src))
assert maks < CFG.FP16_MAX
total = sum(f.stat().st_size for f in CFG.OUT_DIR.glob('*.safetensors')) / 1e9
print(f'{len(kunci_fp16)} tensor cocok | total {total:.2f} GB | |bobot| maks {maks:.2f}')
shutil.rmtree(CFG.HF_CACHE, ignore_errors=True)

#### Insights

> Isi setelah run: jumlah shard, total ukuran, dan nilai bobot terbesar. Bila assert overflow gagal, notebook utama tetap bisa memakai jalur unduhan fp32 langsung.